# Bronze — ERPNext bank GL ingestion

Capture every voucher in the selected batch. See the reconciliation README for manifest setup.

In [3]:
%pip install boto3
%pip install requests

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import hashlib
import json
import os
import sys
from decimal import Decimal
from pathlib import Path
from urllib.parse import urlsplit
from uuid import uuid4

import boto3
import requests
from botocore.config import Config
from botocore.exceptions import ClientError
from dotenv import dotenv_values
from pyspark.sql import SparkSession, functions as F

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

# Support launching Jupyter from either the repository root or the notebook folder.
cwd = Path.cwd().resolve()
candidates = [parent if parent.name == "data-platform" else parent / "data-platform"
              for parent in (cwd, *cwd.parents)]
env_path = next((folder / ".env" for folder in candidates if (folder / ".env").is_file()), None)
if env_path is None:
    raise FileNotFoundError("Create data-platform/.env and launch from the repository.")
cfg = dotenv_values(env_path, interpolate=False)

def required(name):
    value = cfg.get(name)
    if not value:
        raise ValueError(f"Set {name} in data-platform/.env")
    return value

endpoint = cfg.get("MINIO_ENDPOINT") or "http://192.168.29.250:9000"
bucket = cfg.get("BRONZE_BUCKET") or "bronze"
os.environ["AWS_ACCESS_KEY_ID"] = required("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = required("AWS_SECRET_ACCESS_KEY")
os.environ["PYSPARK_PYTHON"] = sys.executable

# Use path-style addressing for both Spark and the raw-object client.
s3 = boto3.client(
    "s3", endpoint_url=endpoint, region_name="us-east-1",
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    config=Config(s3={"addressing_style": "path"}),
)
warehouse_dir = Path.home() / ".cache" / "reconciliation" / "spark-warehouse"
warehouse_dir.mkdir(parents=True, exist_ok=True)
spark = (
    SparkSession.builder.master("local[2]").appName("bronze-erpnext-gl")
    .config("spark.jars.packages",
            "io.delta:delta-spark_2.12:3.3.3,org.apache.hadoop:hadoop-aws:3.3.4")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.hadoop.fs.s3a.endpoint", endpoint)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", str(urlsplit(endpoint).scheme == "https").lower())
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .config("spark.sql.warehouse.dir", warehouse_dir.as_uri())
    .getOrCreate()
)
import re

# Set a manifest path in the kernel environment or data-platform/.env.
manifest_path = os.environ.get("RECON_BATCH_MANIFEST") or cfg.get("RECON_BATCH_MANIFEST")
batch = None
if manifest_path:
    path = env_path.parent / Path(manifest_path).expanduser()
    batch = json.loads(path.read_text(encoding="utf-8-sig"))
    run = batch["run_id"]
    company = batch["company"]
    account = batch["bank_gl_account"]
    vouchers = [item["voucher_no"] for item in batch["vouchers"]]
    if not all(isinstance(v, str) and v.strip() for v in [company, account, *vouchers]):
        raise ValueError("Manifest company, account and voucher IDs must be nonempty strings.")
    if not vouchers or len(vouchers) != len(set(vouchers)):
        raise ValueError("Manifest must contain at least one voucher and no duplicates.")
else:
    run = required("DEMO_RUN_ID")
    company = required("ERPNEXT_COMPANY")
    account = required("ERPNEXT_BANK_ACCOUNT")
    vouchers = [required("ERPNEXT_VOUCHER_NO")]
if not isinstance(run, str) or not re.fullmatch(r"[A-Za-z0-9_-]{1,80}", run):
    raise ValueError("Invalid reconciliation run ID.")
batch_gl_uri = f"s3a://{bucket}/tables/reconciliation/erpnext_gl/{run}"


:: loading settings :: url = jar:file:/home/kirit/.venvs/bronze-py311/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/kirit/.ivy2/cache
The jars for the packages stored in: /home/kirit/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-dfb07ae0-a5a4-4aae-ba9b-3c7fea78a2d4;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.3 in central
	found io.delta#delta-storage;3.3.3 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 361ms :: artifacts dl 15ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.3.3 from central in [default]
	io.delta#delta-storage;3.3.3 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.hadoop

## 1. Verify the MinIO Delta write and read

In [5]:
# Isolate the test objects from the Bronze data.
smoke_path = f"s3a://{bucket}/_smoke/delta-{uuid4().hex}"
spark.range(2).write.format("delta").mode("errorifexists").save(smoke_path)
ids = [row.id for row in spark.read.format("delta").load(smoke_path).orderBy("id").collect()]
if ids != [0, 1]:
    raise RuntimeError("Unexpected Delta smoke-test result.")
print("PASS: wrote and read 2 Delta rows")

26/11/17 06:33:38 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/11/17 06:33:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


PASS: wrote and read 2 Delta rows


## Extract bank GL entries

In [6]:
import json
import re

import requests

run = required("DEMO_RUN_ID")
if not re.fullmatch(r"[A-Za-z0-9_-]{1,80}", run):
    raise ValueError("Invalid DEMO_RUN_ID")

site = required("ERPNEXT_URL").rstrip("/")
company = required("ERPNEXT_COMPANY")
account = required("ERPNEXT_BANK_ACCOUNT")

fields = [
    "name", "company", "voucher_type", "voucher_no", "account",
    "posting_date", "debit", "credit", "is_cancelled",
]
filters = [
    ["company", "=", company],
    ["account", "=", account],
    ["voucher_type", "=", "Journal Entry"],
    ["is_cancelled", "=", 0],
]

# Optional lower bound for the demo. Leave unset to read all posting dates.
if posting_from := cfg.get("ERP_EXTRACT_POSTING_FROM"):
    from datetime import date

    filters.append(["posting_date", ">=", date.fromisoformat(posting_from).isoformat()])

captures = {}
last_name = None

with requests.Session() as session:
    session.headers["Authorization"] = (
        f"token {required('ERPNEXT_API_KEY')}:{required('ERPNEXT_API_SECRET')}"
    )

    while True:
        page_filters = filters.copy()
        if last_name is not None:
            page_filters.append(["name", ">", last_name])

        response = session.get(
            f"{site}/api/resource/GL%20Entry",
            params={
                "fields": json.dumps(fields),
                "filters": json.dumps(page_filters),
                "order_by": "name asc",
                "limit_page_length": 100,
            },
            timeout=30,
            allow_redirects=False,
        )
        if response.status_code != 200:
            raise RuntimeError(f"ERPNext GL discovery returned HTTP {response.status_code}")

        rows = response.json()["data"]
        if not rows:
            break

        for row in rows:
            gl_id = row["name"]
            if gl_id in captures:
                raise RuntimeError(f"ERPNext pagination returned duplicate GL ID {gl_id}")
            captures[gl_id] = (row, response.content)

        last_name = rows[-1]["name"]

if not captures:
    raise RuntimeError(f"No journal GL entries found for {company} / {account}")

print(f"Discovered {len(captures)} GL entries for {company} / {account}")

Discovered 8 GL entries for Demo Finance / Demo Operating Bank - DF


## Land raw captures and publish the Bronze batch

In [7]:
import hashlib

from botocore.exceptions import ClientError
from pyspark.sql import functions as F

records = []
voucher_ids = set()

for gl_id, (row, raw_bytes) in captures.items():
    source_key = hashlib.sha256(json.dumps([site, gl_id]).encode()).hexdigest()
    raw_hash = hashlib.sha256(raw_bytes).hexdigest()
    raw_key = f"raw/erpnext/gl_entry/{source_key}/{raw_hash}.json"

    try:
        s3.put_object(
            Bucket=bucket,
            Key=raw_key,
            Body=raw_bytes,
            ContentType="application/json",
            IfNoneMatch="*",
        )
    except ClientError as exc:
        if exc.response["ResponseMetadata"]["HTTPStatusCode"] != 412:
            raise

    records.append({
        **row,
        "source_key": source_key,
        "source_instance": site,
        "raw_uri": f"s3a://{bucket}/{raw_key}",
        "run_id": run,
    })
    voucher_ids.add(row["voucher_no"])

gl_uri = f"s3a://{bucket}/tables/reconciliation/erpnext_gl/{run}"
frame = spark.read.json(
    spark.sparkContext.parallelize([json.dumps(row) for row in records])
).withColumn("ingested_at", F.current_timestamp())

frame.write.format("delta").mode("overwrite").save(gl_uri)

landed = spark.read.format("delta").load(gl_uri)
if landed.count() != len(records):
    raise RuntimeError(f"Bronze {run}: GL snapshot row count differs from the extract")

gl_manifest = {
    "run_id": run,
    "company": company,
    "bank_gl_account": account,
    "source_instance": site,
    "bronze_gl_uri": gl_uri,
    "gl_row_count": len(records),
    "vouchers": [{"voucher_no": voucher} for voucher in sorted(voucher_ids)],
}
manifest_key = f"raw/reconciliation/{run}/erpnext-gl-manifest.json"
s3.put_object(
    Bucket=bucket,
    Key=manifest_key,
    Body=json.dumps(gl_manifest, indent=2).encode("utf-8"),
    ContentType="application/json",
)

print(f"Bronze GL manifest: s3a://{bucket}/{manifest_key}")
landed.groupBy("voucher_no").count().orderBy("voucher_no").show(truncate=False)

Bronze GL manifest: s3a://bronze/raw/reconciliation/silver-demo-001/erpnext-gl-manifest.json


+-----------------+-----+
|voucher_no       |count|
+-----------------+-----+
|ACC-JV-2026-00001|1    |
|ACC-JV-2026-00002|1    |
|ACC-JV-2026-00003|1    |
|ACC-JV-2026-00004|1    |
|ACC-JV-2026-00005|1    |
|ACC-JV-2026-00006|1    |
|ACC-JV-2026-00007|1    |
|ACC-JV-2026-00008|1    |
+-----------------+-----+



## Verify the published batch

In [8]:
actual = spark.read.format("delta").load(batch_gl_uri)
if actual.count() != len(records) or actual.select("source_key").distinct().count() != len(records):
    raise RuntimeError(f"Bronze {run}: read-back count or source key uniqueness check failed")
actual.groupBy("company", "voucher_no").count().orderBy("company", "voucher_no").show()


+------------+-----------------+-----+
|     company|       voucher_no|count|
+------------+-----------------+-----+
|Demo Finance|ACC-JV-2026-00001|    1|
|Demo Finance|ACC-JV-2026-00002|    1|
|Demo Finance|ACC-JV-2026-00003|    1|
|Demo Finance|ACC-JV-2026-00004|    1|
|Demo Finance|ACC-JV-2026-00005|    1|
|Demo Finance|ACC-JV-2026-00006|    1|
|Demo Finance|ACC-JV-2026-00007|    1|
|Demo Finance|ACC-JV-2026-00008|    1|
+------------+-----------------+-----+

